In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error

# 1. Load the cleaned dataset
df = pd.read_csv("cleaned_employee_dataset.csv")

# 2. Separate features and salary
X = df.drop("Salary_INR", axis=1) 
y = df["Salary_INR"]

# 3. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Selecting features for Decision Tree...")
# 4. Feature Selection
base_model = LinearRegression()
selector = SequentialFeatureSelector(base_model, n_features_to_select=8, direction="forward")
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

selected_features = list(X.columns[selector.get_support()])

print("\nTraining the Decision Tree Regression Model...")
# 5. Create and train the Decision Tree brain (Depth=5)
dt_model = DecisionTreeRegressor(random_state=42, max_depth=5)
dt_model.fit(X_train_selected, y_train)

# 6. Evaluate the model
predictions = dt_model.predict(X_test_selected)
r2 = r2_score(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("--- Decision Tree Results ---")
print(f"R2 Score: {r2:.4f}")
print(f"RMSE    : ₹ {rmse:,.2f}")

# 7. Save the trained model
with open('dt_salary_model.pkl', 'wb') as f:
    pickle.dump({
        'model': dt_model, 
        'selector': selector, 
        'feature_names': selected_features
    }, f)
print("\nSuccess! Decision Tree Model saved as 'dt_salary_model.pkl'.")

Selecting features for Decision Tree...

Training the Decision Tree Regression Model...
--- Decision Tree Results ---
R2 Score: 0.9358
RMSE    : ₹ 126,783.58

Success! Decision Tree Model saved as 'dt_salary_model.pkl'.
